# RI #1: Retrieval levers and answers on your corpus

This notebook picks up where
[`04_bring_your_own_corpus.ipynb`](04_bring_your_own_corpus.ipynb) leaves off. It reads the same
folder and the question set notebook 04 built, then measures two things on your documents:

- which retrieval lever finds the gold passage, among BM25, dense, SPLADE, a BM25 and dense hybrid,
  and dense retrieval with a reranker, each against BM25 with a paired 95% interval; and
- how RI #1's single-pass answering configuration does on your answerable and null questions:
  how often it abstains on a null, how often it answers an answerable question correctly, and
  whether its answers stay grounded in the passages it read.

With `RUN_EXPENSIVE = False` and no folder named, it uses notebook 04's stand-in, ten ContractNLI
contracts, and reads the reference results built on Vector's on-prem models, with no model call.

To run it on your documents:

1. Run notebook 04 on your folder with `RUN_EXPENSIVE = True`, so the question set exists.
2. Set `FOLDER`, `CONFIG_FILE`, and `MAX_DOCUMENTS` below to the values you used there, and keep the
   same model family, because the set is stored under the models that built it.
3. Set `RUN_EXPENSIVE = True` and run the cells in order.

The code is in [`own_corpus.py`](own_corpus.py), next to this notebook.

In [1]:
import byoc
import own_corpus
from aieng.advanced_rag import spend
from aieng.advanced_rag.models import choose_models


# Off reads cached results. On scores the levers locally and answers on your models.
RUN_EXPENSIVE = False
# The folder and configuration file you gave notebook 04.
# None uses notebook 04's ten ContractNLI contracts.
FOLDER = None
CONFIG_FILE = None
MAX_DOCUMENTS = 20
# Send a paid answering step whose estimate is over the confirmation threshold.
CONFIRM = False

MODELS = choose_models(show=True)
documents, is_demo = byoc.read_corpus(FOLDER, limit=MAX_DOCUMENTS)
config = byoc.config_for(CONFIG_FILE, is_demo)
# The set notebook 04 built. This notebook never rebuilds it, whatever the flag says.
result, generated = byoc.generate(documents, config, MODELS, live=False, demo=is_demo)
summary = generated.summary()
print(
    f"{len(documents)} documents; {summary['answerable']} answerable and {summary['null']} null questions; "
    f"scope: {config.scope}"
)

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget: not shown in the committed output
question-set reference artifact, built with Qwen3_8-27B on Vector's cluster at $0
             167 calls, 113,545 prompt + 8,662 completion tokens
             your build: gemini-3.5-flash-lite generates, gpt-6-luna checks, gemini-3.8-flash judges; about 167 calls, est. $0.04 (set RUN_EXPENSIVE = True to build)
10 documents; 27 answerable and 7 null questions; scope: document


## 1. Score the retrieval levers

Five arms run on the shared harness at 1,200-character chunks with a 200-character overlap. BM25 is
the baseline. Dense retrieval uses `BAAI/bge-small-en-v1.5` with its query prompt, SPLADE uses
`prithivida/Splade_PP_en_v1`, the hybrid fuses dense and BM25 rankings with weighted reciprocal rank
fusion at a dense weight of 0.2, and the reranker rescores the dense ranking with
`cross-encoder/ms-marco-MiniLM-L-6-v2`. Every model runs on the CPU here, so scoring makes no model
call. On first use it downloads about 0.6 GB of checkpoints, and a folder of 50 documents takes
about ten minutes on 8 CPUs.

Each row gives the arm's mean, then its difference from BM25 on the same questions with a 95%
paired bootstrap interval over 2,000 resamples. An interval that includes zero means this set
cannot tell the arm from BM25. Notebook 04's cautions apply: generated questions score every arm
higher than human questions would, and they say nothing about which answer model is better.

In [2]:
levers = own_corpus.score_levers(result, generated, documents, live=RUN_EXPENSIVE)
mean, versus = levers["mean"], levers["versus_baseline"]
print(
    f"{levers['questions']} answerable questions over {levers['documents_indexed']} indexed documents"
    f"{', corpus-wide' if levers['corpus_wide'] else ', each question in its own document'}"
)
print(
    f"{'arm':<22}" + "".join(f"{m:>10}{'vs ' + levers['baseline'] + ' [95% interval]':>34}" for m in own_corpus.METRICS)
)
for arm in levers["arms"]:
    row = f"{arm:<22}"
    for metric in own_corpus.METRICS:
        gap = versus[metric].get(arm)
        cell = "baseline" if gap is None else f"{gap['mean_difference']:+.3f} [{gap['low']:+.3f}, {gap['high']:+.3f}]"
        row += f"{mean[metric][arm]:>10.3f}{cell:>34}"
    print(row)

levers: cached scores for the set 6204e405bea4
27 answerable questions over 8 indexed documents, each question in its own document
arm                      ndcg@10       vs bm25@1200 [95% interval]  recall@3       vs bm25@1200 [95% interval]       mrr       vs bm25@1200 [95% interval]
bm25@1200                  0.887                          baseline     0.846                          baseline     0.864                          baseline
dense@1200                 0.817           -0.070 [-0.171, +0.025]     0.827           -0.018 [-0.148, +0.111]     0.815           -0.050 [-0.198, +0.100]
splade@1200                0.933           +0.046 [-0.017, +0.116]     0.914           +0.068 [+0.012, +0.142]     0.944           +0.080 [-0.031, +0.198]
hybrid-rrf-0.2@1200        0.897           +0.010 [+0.000, +0.026]     0.883           +0.037 [+0.000, +0.093]     0.886           +0.022 [+0.000, +0.062]
dense+rerank@1200          0.938           +0.050 [-0.012, +0.119]     0.920           +0.074 

## 2. Answer the questions or abstain

The configuration is RI #1's recommended single pass, from [`PIPELINE.md`](PIPELINE.md): retrieve
10 passages with BM25 (pass `retriever="hybrid"` for the fused first stage), grade them without a
model, put the top 5 in the prompt, and make one generation call. The generator abstains with "not
stated" when the passages do not answer, and an answer that cites no passage counts as an
abstention. The judging model then grades each answerable question for correctness against its
reference and for grounding in the passages the generator read, as two separate verdicts.

With the flag on, each live step prints its call count, how many replies are already cached, and an
estimate before it sends anything. The estimate takes every reply at its full completion cap, so it
is an upper bound. A paid step over 10% of the key's remaining budget, or over $1 when the key
reports no budget, stops with `ConfirmationRequiredError` until you set `CONFIRM = True`. Replies
are cached, so a second run costs nothing.

Null detection counts the null questions the pipeline abstained on (recall), the abstentions that
were on null questions (precision), and the null questions it answered anyway (false-answer rate).
Balanced decision accuracy averages the answer rate on answerable questions and the abstention
rate on null ones, so always answering and always abstaining both score 0.5.

In [3]:
answers = own_corpus.answer_questions(result, generated, documents, MODELS, live=RUN_EXPENSIVE, confirm=CONFIRM)
null = answers["null_detection"]
print(
    f"answered with {answers['models']['generation']}, judged with {answers['models']['judging']}, first stage {answers['retriever']}"
)
print(
    f"null questions: {answers['null']}; abstain recall {null['abstain_recall']:.3f}, "
    f"precision {null['abstain_precision']:.3f}, false-answer rate {null['false_answer_rate']:.3f}"
)
print(
    f"answerable questions: {answers['answerable']}; abstained on {answers['answerable_abstained']}, "
    f"correct {answers['correct']} of {answers['judged']} judged, grounded {answers['grounded']}"
)
print(f"balanced decision accuracy {answers['balanced_decision_accuracy']:.3f}")
for step, calls in answers["calls"].items():
    print(f"{step}: {calls['calls']} calls to {calls['model']}")

answers: reference result, built with Qwen3_6-35B-A3B on Vector's cluster at $0 (set RUN_EXPENSIVE = True to answer with your models)
answered with Qwen3_6-35B-A3B, judged with Qwen3_6-35B-A3B, first stage bm25
null questions: 7; abstain recall 0.571, precision 1.000, false-answer rate 0.429
answerable questions: 27; abstained on 0, correct 25 of 27 judged, grounded 27
balanced decision accuracy 0.786
generation: 34 calls to Qwen3_6-35B-A3B
judging: 27 calls to Qwen3_6-35B-A3B


Read the questions the pipeline got wrong before you change anything. A null question it answered
is either a near miss whose changed detail the passages do not catch, or a question the generator
answered from its own knowledge, which the grounding verdict flags. An answerable question it
abstained on usually means the gold passage was not in the top 5, which the lever table above
measures.

In [4]:
rows = answers["rows"]
wrong = [r for r in rows if ("null" in r["kind"]) != r["abstained"]]
print(f"{len(wrong)} of {len(rows)} decisions disagree with the label")
for r in wrong[:4]:
    label = "null" if "null" in r["kind"] else "answerable"
    print(f"- [{label}] {r['question']}\n    -> {r['answer'][:160]}")

3 of 34 decisions disagree with the label
- [null] How long do the obligations in clauses 4 and 5 last?
    -> The undertakings in clauses 2 and 3 will continue in force indefinitely. Clauses 4 and 5 define exceptions to these undertakings and permissions for disclosure,
- [null] Who is permitted to conduct the audit if the Recipient suspects a violation?
    -> The Discloser is permitted to conduct the audit by itself or by an authorized representative.
- [null] For how long do the confidentiality obligations and restrictions remain in effect after the NDA expires or is terminated if the information is a trade secret?
    -> five years


## 3. What to try next

- If an arm beats BM25 with an interval clear of zero, make it the answering first stage. The
  `retriever` argument takes `"bm25"` and `"hybrid"`, and `own_corpus.py` shows where to add
  another.
- If the false-answer rate is high, read those answers first. Requiring citations and a stricter
  prompt are the levers notebook 05 measures.
- Grow the set before trusting a small difference: raise `MAX_DOCUMENTS` or
  `passages_per_document` in notebook 04 and rebuild it.

The last cell prints what this run spent.

In [5]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
